## Exercise 4

### Part 1

In [ ]:
import random
import heapq
import statistics
import math

# Define parameters
M = 10 # Number of servers
mean_service = 8
mean_interarrival = 1.0

num_customers = 10000
num_runs = 10


def sim_run():

    # Simulation clock
    current_time = 0.0

    # The statistics
    arrivals = 0
    blocked = 0

    # Set the numbers of busy servers in the beginning to 0
    busy_servers = 0

    # Event list (time, event type)
    event_list = []

    # First arrival
    first_arrival = random.expovariate(1 / mean_interarrival)
    heapq.heappush(event_list, (first_arrival, "arrival"))

    while arrivals < num_customers:

        # Get the next event
        event_time, event_type = heapq.heappop(event_list)

        current_time = event_time

        if event_type == "arrival":

            arrivals += 1

            # Schedule next arrival
            next_arrival = (
                current_time +
                random.expovariate(1 / mean_interarrival)
            )

            heapq.heappush(
                event_list,
                (next_arrival, "arrival")
            )

            # Check if a server is free
            if busy_servers < M:

                busy_servers += 1

                # Generate the service time
                service_time = random.expovariate(
                    1 / mean_service
                )

                departure_time = (
                    current_time + service_time
                )

                heapq.heappush(
                    event_list,
                    (departure_time, "departure")
                )

            else:
                blocked += 1

        elif event_type == "departure":

            busy_servers -= 1

    return blocked / arrivals


# Run 10 replications
results = []

for _ in range(num_runs):
    results.append(sim_run())

# Mean blocking probability
mean_blocking = statistics.mean(results)

# Standard deviation
std_dev = statistics.stdev(results)


# 95% confidence interval
t_value = 2.262  # t(0.975,9) - 95% confidence interval with 9 degrees of freedom

margin = t_value * std_dev / math.sqrt(num_runs)

lower = mean_blocking - margin
upper = mean_blocking + margin

print("Blocking probabilities:")
for r in results:
    print(f"{r:.5f}")

print()
print(f"Mean = {mean_blocking:.5f}")
print(f"95% CI = [{lower:.5f}, {upper:.5f}]")

Blocking probabilities:
0.12820
0.12010
0.12410
0.12200
0.11340
0.12170
0.12190
0.11610
0.11860
0.12400

Mean = 0.12101
95% CI = [0.11798, 0.12404]


In [14]:
# Checking with the formula
def erlang_b(m, A):
    B = 1.0  # Start with B(0, A) = 1

    for i in range(1, m + 1):
        B = (A * B) / (A * B + i)

    return B


# Set parameters for this case
m = 10
A = 8

blocking_prob = erlang_b(m, A)

print(f"Erlang B blocking probability: {blocking_prob:.6f}")

Erlang B blocking probability: 0.121661


### Part 2

In [ ]:
# Erlang distribution

# Parameters
K = 2
erlang_mean = 1.0

# Erlang arrival process
def interarrival_erlang():
    return random.gammavariate(K, erlang_mean / K)

# Simulation
def sim_erlang():

    busy_servers = 0
    arrivals = 0
    blocked = 0

    event_list = []

    # First arrival
    heapq.heappush(event_list, (interarrival_erlang(), "arrival"))

    while arrivals < num_customers:

        time, event = heapq.heappop(event_list)

        if event == "arrival":

            arrivals += 1

            # Schedule next arrival
            heapq.heappush(
                event_list,
                (time + interarrival_erlang(), "arrival")
            )

            # Service logic
            if busy_servers < M:
                busy_servers += 1

                service_time = random.expovariate(1.0 / mean_service)
                departure_time = time + service_time

                heapq.heappush(
                    event_list,
                    (departure_time, "departure")
                )
            else:
                blocked += 1

        else:
            busy_servers -= 1

    return blocked / arrivals

# Run
results = []

for _ in range(num_runs):
    results.append(sim_erlang())

mean_blocking = statistics.mean(results)
std_blocking = statistics.stdev(results)

# 95% confidence interval with 9 degress of freedom
t_value = 2.262
margin = t_value * std_blocking / math.sqrt(num_runs)

# Results:
print("Erlang arrival case:")
print("Runs:", [round(r, 5) for r in results])
print(f"Mean blocking probability: {mean_blocking:.5f}")
print(f"95% CI: [{mean_blocking - margin:.5f}, {mean_blocking + margin:.5f}]")


Erlang arrival case:
Runs: [0.0914, 0.094, 0.0908, 0.0918, 0.0883, 0.0929, 0.0952, 0.0934, 0.0961, 0.0924]
Mean blocking probability: 0.09263
95% CI: [0.09102, 0.09424]


In [16]:
# Hyper exponential inter arrival times

# Hyper exponential arrival process
def interarrival_hyperexp():
    # With probability p1 choose slow rate, otherwise choose fast rate
    if random.random() < 0.8:
        return random.expovariate(0.8333)
    else:
        return random.expovariate(5.0) 

# Simulation
def sim_hyperexp():

    busy_servers = 0
    arrivals = 0
    blocked = 0

    event_list = []

    # First arrival
    heapq.heappush(event_list, (interarrival_hyperexp(), "arrival"))

    while arrivals < num_customers:

        time, event = heapq.heappop(event_list)

        if event == "arrival":

            arrivals += 1

            # Schedule next arrival
            heapq.heappush(
                event_list,
                (time + interarrival_hyperexp(), "arrival")
            )

            # Service logic
            if busy_servers < M:
                busy_servers += 1

                service_time = random.expovariate(1.0 / mean_service)
                departure_time = time + service_time

                heapq.heappush(
                    event_list,
                    (departure_time, "departure")
                )
            else:
                blocked += 1

        else:
            busy_servers -= 1

    return blocked / arrivals


# Run
results = []

for _ in range(num_runs):
    results.append(sim_hyperexp())

mean_blocking = statistics.mean(results)
std_blocking = statistics.stdev(results)

# 95% confidence interval with 9 degrees of freedom
t_value = 2.262
margin = t_value * std_blocking / math.sqrt(num_runs)

# Results
print("Hyper-exponential arrival case:")
print("Runs:", [round(r, 5) for r in results])
print(f"Mean blocking probability: {mean_blocking:.5f}")
print(f"95% CI: [{mean_blocking - margin:.5f}, {mean_blocking + margin:.5f}]")

Hyper-exponential arrival case:
Runs: [0.1374, 0.137, 0.1307, 0.1431, 0.1331, 0.1433, 0.1503, 0.1463, 0.1393, 0.1408]
Mean blocking probability: 0.14013
95% CI: [0.13588, 0.14438]


### Part 3

In [ ]:
# Poisson arrivals
def interarrival_poisson():
    return random.expovariate(1.0)

# Constant service time
def service_constant():
    return mean_service

# Pareto service times
def service_pareto(k, mean=8.0):
    xm = mean * (k - 1) / k   # Ensures correct mean when k > 1
    u = random.random()
    return xm / (u ** (1.0 / k))

# Uniform service time
def service_uniform():
    return random.uniform(4, 12)  # Mean = 8

# Discrete event simulation
def simulate(service_fn):

    busy = 0
    arrivals = 0
    blocked = 0

    event_list = []

    # First arrival
    heapq.heappush(event_list, (interarrival_poisson(), "arrival"))

    while arrivals < num_customers:

        time, event = heapq.heappop(event_list)

        if event == "arrival":

            arrivals += 1

            # Schedule next arrival
            heapq.heappush(
                event_list,
                (time + interarrival_poisson(), "arrival")
            )

            # Service logic (blocking system)
            if busy < M:
                busy += 1

                service_time = service_fn()
                departure_time = time + service_time

                heapq.heappush(
                    event_list,
                    (departure_time, "departure")
                )
            else:
                blocked += 1

        else:
            busy -= 1

    return blocked / arrivals

# Run experiment
def run_experiment(service_fn, label):

    results = []

    for _ in range(num_runs):
        results.append(simulate(service_fn))

    mean = statistics.mean(results)
    std = statistics.stdev(results)

    t_value = 2.262  # 95% confidence interval with 9 degrees of freedom
    margin = t_value * std / math.sqrt(num_runs)

    print("\n==============================")
    print(label)
    print("==============================")
    print("Runs:", [round(r, 5) for r in results])
    print(f"Mean blocking: {mean:.5f}")
    print(f"95% CI: [{mean - margin:.5f}, {mean + margin:.5f}]")

# Main
if __name__ == "__main__":

    # Constant service time
    run_experiment(service_constant, "Constant service time")

    # Pareto k = 1.05 (very heavy tail)
    run_experiment(lambda: service_pareto(1.05), "Pareto k = 1.05")

    # Pareto k = 2.05 (less heavy tail)
    run_experiment(lambda: service_pareto(2.05), "Pareto k = 2.05")

    # Uniform service time
    run_experiment(service_uniform, "Uniform service time")


Constant service time
Runs: [0.127, 0.1119, 0.1202, 0.1195, 0.1244, 0.1193, 0.1189, 0.1215, 0.117, 0.1117]
Mean blocking: 0.11914
95% CI: [0.11570, 0.12258]

Pareto k = 1.05
Runs: [0.0015, 0.0004, 0.0006, 0.0048, 0.0004, 0.0024, 0.0004, 0.0014, 0.0003, 0.0004]
Mean blocking: 0.00126
95% CI: [0.00024, 0.00228]

Pareto k = 2.05
Runs: [0.1215, 0.1187, 0.1315, 0.1176, 0.1194, 0.1119, 0.1175, 0.1158, 0.1206, 0.1229]
Mean blocking: 0.11974
95% CI: [0.11605, 0.12343]

Uniform service time
Runs: [0.1112, 0.118, 0.1231, 0.1196, 0.1197, 0.1118, 0.124, 0.1184, 0.1205, 0.1172]
Mean blocking: 0.11835
95% CI: [0.11535, 0.12135]


### Part 4

The different confidence intervals show that is approximately stable for systems with similar cariability characteristics (Part 1 and most of Part 3). Here, the intervals overlap, and they indicate no stitistically significant difference.

Changing the arrival process, however, seems to have a strong impact. The Erlang arrivals process results in a significantly lower blocking probability, due to reduced variability, while the hyper-exponential arrivals have a significantly higher blocking probability, since bursty arrivals can cause temporary overload of the servers.

The very heavy-tailed (k=1.05) Pareto case has a drastically reduced blocking probability, which reflects the dominance of rare but very large service times and increated system irregularity.